## Example of the `aitlas` toolbox in the context of image segmentation
---
```
Author: Ana Kostovska
Organisation: Bias Variance Labs
Website: https://www.bvlabs.ai/
Ljubljana, 2024
```
---

### Importing required packages

In [1]:
!conda env list

# conda environments:
#
base                     C:\ProgramData\miniconda3
adaf39                *  C:\ProgramData\miniconda3\envs\adaf39
adam-ml                  C:\ProgramData\miniconda3\envs\adam-ml
aitlas                   C:\ProgramData\miniconda3\envs\aitlas
amenai                   C:\ProgramData\miniconda3\envs\amenai
amenbist                 C:\ProgramData\miniconda3\envs\amenbist
amenreg38                C:\ProgramData\miniconda3\envs\amenreg38
geoprocessing            C:\ProgramData\miniconda3\envs\geoprocessing
las                      C:\ProgramData\miniconda3\envs\las



In [16]:
from pathlib import Path

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
from aitlas.datasets import TiiLIDARDatasetSegmentation
from aitlas.models import HRNet
model_config = TiiLIDARDatasetSegmentation.get_fixed_model_config()

### Loading train, validation and test data

Input parameters for train, validation and test data:

- **batch_size**: The number of samples processed before the model is updated. A larger batch size can speed up processing but requires more memory.
- **num_workers**: The number of worker processes that will be used for processing data. Increasing the number of workers can significantly speed up data processing, however, it also increases memory and CPU/GPU usage.
- **object_class**: A parameter that specifies the type of archaeological object you are interested in processing, e.g., 'AO', 'barrow', 'enclosure', 'ringfort'.
- **object_class_band_id**: An integer parameter identifying the band where the annotations for a specific object class are located within the segmentation masks.
- **visualisation_type**: The vizuelization type used for the patches, e.g., 'SLRM'.
- **DFM_quality**: List of annotation qualities to be included in the processed data, e.g., '1,2'.
- **keep_empty_patches**: A boolean parameter that controls if empty patches are kept. Set to False when training since "empty" data can't be used for training. For testing or validation, True can be used to check how the model handles empty patches.
- **shuffle**: Determines whether the data should be shuffled before being processed. 
- **data_dir**: The directory path where the input data is stored. 
- **annotations_dir**: The directory path where the segmentation masks are stored. 
- **transforms**: A list of transformations applied to the input data during processing.
- **target_transforms**: A list of transformations applied to the segmentation masks during processing.
- **joint_transforms**: Transformations applied simultaneously to both the input data and segmentation masks.

In [39]:
batch_size = 16
num_workers = 4
object_class = "barrow"
object_class_band_id = 0
visualisation_type = "SLRM"

In [43]:
train_data = r"r:\delovno\nejc\training_samples_BiH_v31\samples\train"
train_mask = r"r:\delovno\nejc\training_samples_BiH_v31\labels\segmentation_masks\train"

validation_data = r"r:\delovno\nejc\training_samples_BiH_v31\samples\validation"
validation_mask = r"r:\delovno\nejc\training_samples_BiH_v31\labels\segmentation_masks\validation"

test_data = r"r:\delovno\nejc\training_samples_BiH_v31\samples\test"
test_mask = r"r:\delovno\nejc\training_samples_BiH_v31\labels\segmentation_masks\test"

In [30]:
train_data = r"r:\delovno\nejc\test_adaf_retrain\samples\train"
train_mask = r"r:\delovno\nejc\test_adaf_retrain\labels\segmentation_masks\train"

validation_data = r"r:\delovno\nejc\test_adaf_retrain\samples\validation"
validation_mask = r"r:\delovno\nejc\test_adaf_retrain\labels\segmentation_masks\validation"

test_data = r"r:\delovno\nejc\test_adaf_retrain\samples\test"
test_mask = r"r:\delovno\nejc\test_adaf_retrain\labels\segmentation_masks\test"

In [44]:
train_dataset_config = {
    "batch_size": batch_size,
    "num_workers": num_workers,
    "object_class": object_class,
    "object_class_band_id": object_class_band_id,
    "visualisation_type": visualisation_type,
    "DFM_quality": '1',
    "shuffle": True,
    "keep_empty_patches": False,
    "data_dir": train_data,
    "annotations_dir": train_mask,
    "joint_transforms": ["aitlas.transforms.FlipHVRandomRotate"],
    "transforms": ["aitlas.transforms.Transpose"],
	"target_transforms": ["aitlas.transforms.Transpose"]
}
train_dataset = TiiLIDARDatasetSegmentation(train_dataset_config)

validation_dataset_config = {
    "batch_size": batch_size,
    "num_workers": num_workers,
    "object_class": object_class,
    "object_class_band_id": object_class_band_id,
    "visualisation_type": visualisation_type,
    "DFM_quality": '1',
    "shuffle": False,
    "keep_empty_patches": False,
    "data_dir": validation_data,
    "annotations_dir": validation_mask,
    "transforms": ["aitlas.transforms.Transpose"],
    "target_transforms": ["aitlas.transforms.Transpose"]
}
validation_dataset = TiiLIDARDatasetSegmentation(validation_dataset_config)

test_dataset_config = {
    "batch_size": batch_size,
    "num_workers": num_workers,
    "object_class": object_class,
    "object_class_band_id": object_class_band_id,
    "visualisation_type": visualisation_type,
    "DFM_quality": '1',
    "shuffle": False,
    "keep_empty_patches": False,
    "data_dir": test_data,
    "annotations_dir": test_mask,
    "transforms": ["aitlas.transforms.Transpose"],
	"target_transforms": ["aitlas.transforms.Transpose"]
}
test_dataset = TiiLIDARDatasetSegmentation(test_dataset_config)

len(train_dataset), len(validation_dataset), len(test_dataset)

(0, 0, 0)

In [38]:
train_dataset.get_labels

<bound method SemanticSegmentationDataset.get_labels of <aitlas.datasets.tii_lidar_binary_with_preprocessing.TiiLIDARDatasetSegmentation object at 0x000001766490D340>>


In [13]:
dir(train_dataset)

['DFM_quality',
 '__abstractmethods__',
 '__add__',
 '__class__',
 '__class_getitem__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__orig_bases__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_is_protocol',
 'apply_transformations',
 'batch_size',
 'color_mapping',
 'config',
 'data_distribution_barchart',
 'data_distribution_table',
 'dataloader',
 'get_fixed_model_config',
 'get_labels',
 'get_name',
 'images',
 'joint_transform',
 'keep_empty_patches',
 'labels',
 'load_dataset',
 'load_transforms',
 'masks',
 'name',
 'num_workers',
 'object_class',
 'object_class_band_id',
 'pin_memory',
 'prepare',
 'process_single_mask',
 'schema',
 'sho

### Model creation

In [26]:
model = HRNet(model_config)
model.prepare()

2025-11-11 10:53:24,005 INFO Loading pretrained weights from Hugging Face hub (timm/hrnet_w48.ms_in1k)
2025-11-11 10:53:24,269 INFO HTTP Request: HEAD https://huggingface.co/timm/hrnet_w48.ms_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
2025-11-11 10:53:24,272 INFO [timm/hrnet_w48.ms_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.


### Loading pretrained ADAF model (optional)

If you don't want to use an existing model, you can skip this step. If you do want to use one, set the model path, uncomment the lines, and run the cell to load the model into memory.

In [27]:
model_path = r".\adaf\ml_models\barrow_HRNet_SLRM_512px_pretrained_train_12_val_124_with_Transformation.tar" 
model.load_model(model_path)

2025-11-11 10:53:26,368 INFO Loading checkpoint .\adaf\ml_models\barrow_HRNet_SLRM_512px_pretrained_train_12_val_124_with_Transformation.tar
2025-11-11 10:53:28,440 INFO Loaded checkpoint .\adaf\ml_models\barrow_HRNet_SLRM_512px_pretrained_train_12_val_124_with_Transformation.tar at epoch 25


(25,
 0.013847780594127303,
 1698762193,
 'train_barrow_HRNet_SLRM_512px_pretrained_train_12_val_124_Transformation')

### Training the model

Input parameters: 
- **epochs**: The total number of training cycles the model will undergo. Each epoch represents one complete pass of the training dataset through the model.
- **model_directory**: Path to the directory where the trained model and its checkpoints will be saved. This is used for storing the model during and after training.
- **run_id**: Name of the subdirectory within the model_directory to store results from different runs

In [28]:
epochs = 20
model_directory = r"r:\delovno\nejc\models"
run_id = 'barrow_stone_v3'

In [29]:
model.train_and_evaluate_model(
    train_dataset=train_dataset,
    val_dataset=validation_dataset,
    epochs=epochs,
    model_directory=model_directory,
    run_id=run_id
)

2025-11-11 10:53:34,283 INFO Starting training.


ValueError: num_samples should be a positive integer value, but got num_samples=0

### Model evaluation

In [ ]:
model = HRNet(model_config)
model.prepare()
model.running_metrics.reset()
model_path = r"./models/semantic_segmentation/best_checkpoint_test.pth.tar" # update the path!
model.evaluate(dataset=test_dataset, model_path=model_path)
model.running_metrics.get_scores(model.metrics)